# Basic Task 1: Create DataFrame from In-Memory Data
**Goal:** Create a DataFrame from a Python list/dictionary and display it.

In [0]:
# Create a DataFrame from a Python list of dictionaries
data = [
    {
        "product_id": 101,        # ← fixed: was "productId"
        "product_name": "Wireless Mouse",
        "price": 25.99,
        "category": "Electronics"
    },
    {
        "product_id": 102,
        "product_name": "Notebook",
        "price": 5.49,
        "category": "Stationery"
    },
    {
        "product_id": 103,
        "product_name": "Coffee Mug",
        "price": 12.00,
        "category": "Kitchen"
    }
]

# Create DataFrame using SparkSession (already available as 'spark' in Databricks)
df_products = spark.createDataFrame(data)

# Display the DataFrame
display(df_products)

## Basic Task 2: Read CSV with inferSchema vs Explicit StructType

**Goal:** Read the same CSV file two different ways and compare the schemas.

- **Method 1:** Let Spark guess the column types (`inferSchema=True`)
- **Method 2:** We tell Spark exactly what each column type is (`StructType`)

In [0]:
%sql
-- Create a dedicated schema inside the cyntexa_dev catalog
CREATE SCHEMA IF NOT EXISTS cyntexa_dev.sparkbasicsday4;

-- Use the new schema
USE cyntexa_dev.sparkbasicsday4;

In [0]:
# Read CSV with header=True and inferSchema=True
df_inferred = spark.read\
    .format("csv")\
    .option("header" , "true")\
    .option("inferSchema" , "true")\
    .load("/Volumes/cyntexa_dev/sales/external_data/Employees/")

# Print tree representation of the schema
df_inferred.printSchema()

In [0]:
display(df_inferred)

In [0]:
from pyspark.sql.types import(
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType
)

# Define explicit StructType schema
explicit_schema = StructType(
    [
        StructField("id",IntegerType(),True),
        StructField("name",StringType(),True),
        StructField("department",StringType(),True),
        StructField("salary",DoubleType(),True),
        StructField("join_date",StringType(),True)  # Enforced as StringType
    ]
)

# Read CSV with header=True and pass the schema defined above
df_explicit = spark.read\
    .format("csv")\
    .option("header","true")\
    .schema(explicit_schema)\
    .load("/Volumes/cyntexa_dev/sales/external_data/Employees/")

# Print tree representation of the schema
df_explicit.printSchema()

Here, we bypass the automated scanning step by explicitly specifying the exact data types for each field. Notice how `join_date` is explicitly forced to be a `StringType` instead of a date.

In [0]:
print("Inferred Schema Fields:")
print(df_inferred.schema)
print("Explicit Schema Fields:")
print(df_explicit.schema)

# Print comparison highlight
print("\n--- Key Differences ---")
for f_inf , f_exp in zip(df_inferred.schema, df_explicit.schema):
    if f_inf.dataType != f_exp.dataType:
        print(f"Column '{f_inf.name}': Inferred as {f_inf.dataType}, Explicitly set as {f_exp.dataType}")

## Explanation

- **Performance:** `inferSchema=True` requires an extra pass over the dataset to guess the data types, which causes performance overhead on large files. Providing an explicit `StructType` skips this extra pass completely.

- **Type Control:** `inferSchema` parsed `join_date` as a `DateType`, whereas the explicit schema enforced `StringType`. Explicit schemas protect production pipelines against schema drift or misparsed data.


## Basic Task 3: Filter Transformation & Lazy Evaluation


In [0]:
# Step 1: Transformation (Lazy Evaluation)
df_filtered = df_explicit.filter(df_explicit["salary"] > 60000)

print("Transformation defined. Notice that no Spark job has run yet in the cluster UI!")

# Step 2: Action (Triggers Execution)
print(f"\nFiltered Record Count: {df_filtered.count()}")
df_filtered.show()

## Lazy Evaluation vs Actions

### Transformations (Lazy)
Operations like `.filter()`, `.select()`, `.groupBy()` only build an **execution plan**.  
Nothing runs on the cluster until an **action** is called.

### Actions (Triggers Execution)
Operations like `.count()`, `.show()`, `.collect()`, `.write()` tell Spark to actually run the plan.

### Why Lazy?
- **Optimization**: Spark looks at the full plan and optimizes it before running.
- **Efficiency**: Avoids unnecessary reads and computations.

### Example
```python
df_filtered = df.filter(df["salary"] > 60000)   # Lazy: just a plan
df_filtered.count()                              # Action: runs now!
df_filtered.show()                               # Action: runs again!

⚠️ Note: Each action triggers a full re-run. Use .cache() to avoid re-computation.

### 2. Intermediate Tasks 
## Task - 4 

In [0]:
from pyspark.sql.functions import current_date , col

# -----------------------------------------
# 1. READ the CSV file from the Volume
# -----------------------------------------

df_emp = spark.read\
    .format("csv")\
    .option("header" , "true")\
    .option("inferSchema" , "true")\
    .load("/Volumes/cyntexa_dev/spark_basics_day4/raw/Employees/")

df_emp.show()

In [0]:
# -----------------------------------------
# 2. FILTER the data
# -----------------------------------------

# Let's keep only employees who are ACTIVE and earn MORE than 50,000
df_emp_filtered = df_emp.filter((col("is_active") == "true") & (col("salary") > 50000))
df_emp_filtered.show()

In [0]:
# -----------------------------------------
# 3. ADD a new column: ingestion_date
# -----------------------------------------

# This records WHEN the data was processed. It's a common practice in data pipelines.
df_transformed = df_emp_filtered.withColumn("ingestion_date" , current_date())
df_transformed.show()

# -----------------------------------------
# 4. WRITE as Delta Table
# -----------------------------------------
# We save the result into your schema as a Delta table.
# mode("overwrite") means: if the table exists, replace it.

df_transformed.write\
    .mode("overwrite")\
    .saveAsTable("cyntexa_dev.spark_basics_day4.employees_processed")

## Task 5

In [0]:
from pyspark.sql.functions import explode , col

df_json = spark.read\
        .option("multiLine" , "true")\
        .json("/Volumes/cyntexa_dev/spark_basics_day4/raw/Employees JSON /")

# df_json.printSchema()  # Shows the structure
df_json.show(5 , truncate=False)

In [0]:
# -----------------------------------------
# 2. FLATTEN using Dot Notation (for nested objects)
# -----------------------------------------

# The 'contact' column has nested fields: email, phone, address.city, etc.
# We can access them directly using dot notation.
df_flat_contact = df_json.select(
    col("emp_id"),
    col("name"),
    col("department"),
    col("contact.email").alias("email"),  # dot notation!
    col("contact.phone").alias("phone"),  # dot notation!
    col("contact.address.city").alias("city"), # nested dot notation!
    col("contact.address.state").alias("state"),
    col("contact.address.zip").alias("zip")
)

df_flat_contact.show(5 , truncate=False)

In [0]:
# -----------------------------------------
# 3. FLATTEN using explode() (for nested arrays)
# -----------------------------------------

# The 'projects' column is an ARRAY. Each employee has multiple projects.
# explode() turns each project into its own row.

df_exploded = df_json.select(
    col("emp_id"),
    col("name"),
    explode(col("projects")).alias("project") # explode the array!
)

# df_exploded.show(5 , truncate = False)

# Now 'project' is still a struct, so we flatten it further
df_projects_flat = df_exploded.select(
    col("emp_id"),
    col("name"),
    col("project.name").alias("project_name"),
    col("project.status").alias("project_status"),
    col("project.hours").alias("project_hours")
)

df_projects_flat.show(5 , truncate=False)

In [0]:
# -----------------------------------------
# 4. Explode the 'skills' array too
# -----------------------------------------

df_skills = df_json.select(
    col("emp_id"),
    col("name"),
    explode(col("skills")).alias("skill")
)

df_skills.show(5 , truncate=False)

## Task 6

In [0]:
from pyspark.sql.functions import col , sum as spark_sum , avg , count 

# -----------------------------------------
# 1. Build a Multi-Step Transformation Chain
# -----------------------------------------

df = spark.read\
    .option("header" , "true")\
    .option("inferSchema" , "true")\
    .csv("/Volumes/cyntexa_dev/spark_basics_day4/raw/Employees/")

# Chain 1: Simple filtering and adding a column (NO shuffle)
df_step1 = df.filter(col("is_active") == "true")\
            .withColumn("salary_bonus" , col("salary") * 1.10)

# Chain 2: GroupBy aggregation (THIS CAUSES A SHUFFLE!)
df_step2 = df_step1.groupBy("department")\
            .agg(
                spark_sum("salary").alias("total_salary"),
                avg("salary").alias("avg_salary"),
                count("*").alias("employee_count")
            )
# Chain 3: Filter the aggregated results (NO shuffle)
df_step3 = df_step2.filter(col("employee_count") > 1)

# Chain 4: Join with another DataFrame (THIS CAUSES A SHUFFLE!)
# Let's create a small lookup table for department budgets

dept_data = [("Engineering", 500000), ("Sales", 300000), ("HR", 150000)]
dept_col = ["department", "budget"]

df_dept = spark.createDataFrame(dept_data , dept_col)

df_final = df_step3.join(df_dept , on = "department" , how = "inner")

# -----------------------------------------
# 2. Call .explain() to see the Physical Plan
# -----------------------------------------
print("=" * 80)
print("PHYSICAL PLAN (Extended):")
print("=" * 80)
df_final.explain(extended=True)


# PySpark Multi-Step Transformation Chain 

---

## 1. What is a "Shuffle"? (The Core Concept)

### Simple Definition
> **Shuffle = Moving data across the network from one executor to another.**

When Spark runs on a cluster, your data is split into chunks and each chunk lives on a different machine (called an **executor**). A shuffle happens when Spark needs to **reorganize** that data so rows with the **same key** end up on the **same executor**.

### Why is Shuffle expensive?

| Problem | What happens |
|---------|-------------|
| **Network I/O** | Data travels over the wire between machines |
| **Disk I/O** | Spark spills intermediate data to disk when memory is full |
| **Slower** | Your job pauses until all data is redistributed |
| **GC Pressure** | More objects in memory = more garbage collection |

### Visual: Before vs After Shuffle

```
BEFORE (3 Executors)                     AFTER (Grouped by Key)
┌─────────────────────┐                  ┌─────────────────────┐
│  Executor 1         │                  │  Engineering Group  │
│  Eng: 50K           │      SHUFFLE     │  50K, 60K, 55K, 70K │
│  Sales: 30K         │  ═════════════►  │                     │
│  Eng: 60K           │                  ├─────────────────────┤
└─────────────────────┘                  │  Sales Group        │
┌─────────────────────┐                  │  30K, 35K, 28K      │
│  Executor 2         │                  │                     │
│  HR: 20K            │  ═════════════►  ├─────────────────────┤
│  Eng: 55K           │                  │  HR Group           │
│  Sales: 35K         │                  │  20K, 25K           │
└─────────────────────┘                  └─────────────────────┘
┌─────────────────────┐
│  Executor 3         │
│  HR: 25K            │  ═════════════►
│  Sales: 28K         │
│  Eng: 70K           │
└─────────────────────┘
```

> **Key takeaway:** Shuffle is the "data traffic jam" of Spark. Avoid it when possible.

---

## 2. Narrow vs Wide Transformations

| Type | Operations | Shuffle? | Why? |
|------|-----------|----------|------|
| **Narrow** | `filter()`, `withColumn()`, `select()`, `drop()`, `where()` | ❌ **NO** | Each row is processed independently. Executor A does not need data from Executor B. |
| **Wide** | `groupBy()`, `join()`, `repartition()`, `orderBy()`, `distinct()` | ✅ **YES** | Rows with the same key must be brought together from different executors. |

### Rule of Thumb
> If an operation needs to look at **multiple rows together** (e.g., sum, average, match two tables), it is a **Wide Transformation** and will likely shuffle.

---

## 3. Step-by-Step Breakdown

### Step 0 — `spark.read.csv(...)`
- Reads the CSV file in a distributed way.
- Each executor reads its own slice of the file.
- **No shuffle.**

### Step 1 — `filter()` + `withColumn()`
- **What it does:** Keeps only active employees and adds a 10% bonus column.
- **Shuffle?** ❌ No.
- **Why?** Every row is checked on its own. Row 1 does not care about Row 2.

### Step 2 — `groupBy()` + `agg()`
- **What it does:** Groups by department and calculates total salary, average salary, and employee count.
- **Shuffle?** ✅ **YES.**
- **Why?** To calculate `sum(salary)` for Engineering, Spark must collect **all Engineering rows** from every executor into one place.

### Step 3 — `filter()` on aggregated data
- **What it does:** Removes departments with 1 or fewer employees.
- **Shuffle?** ❌ No.
- **Why?** After grouping, each department already lives on a single executor. The filter is local.

### Step 4 — Create `df_dept`
- **What it is:** A tiny lookup table (dimension table) with only 3 rows.
- It tells us the **budget** for each department.

```
df_dept:
┌─────────────┬────────┐
│ department  │ budget │
├─────────────┼────────┤
│ Engineering │ 500000 │
│ Sales       │ 300000 │
│ HR          │ 150000 │
└─────────────┴────────┘
```

### Step 5 — `join(df_dept, on="department")`
- **What it does:** Merges the employee summary with the department budget table.
- **Shuffle?** ✅ **YES.**
- **Why?** Spark must bring matching departments from both tables to the same executor so they can be merged.

#### How the Join Works (Inner Join)

```
Left Table (df_step3)          Right Table (df_dept)         Result (df_final)
┌─────────────┬──────────┐     ┌─────────────┬────────┐     ┌─────────────┬──────────┬────────┐
│ department  │ total_sal│     │ department  │ budget │     │ department  │ total_sal│ budget │
├─────────────┼──────────┤  +  ├─────────────┼────────┤  =  ├─────────────┼──────────┼────────┤
│ Engineering │ 235000   │     │ Engineering │ 500000 │     │ Engineering │ 235000   │ 500000 │
│ Sales       │ 93000    │     │ Sales       │ 300000 │     │ Sales       │ 93000    │ 300000 │
└─────────────┴──────────┘     │ HR          │ 150000 │     └─────────────┴──────────┴────────┘
                               └─────────────┴────────┘
```

> **Note:** HR does **not** appear in the result because it was already filtered out in Step 3 (`employee_count > 1`).

---

## 4. Pro Tip: Use `broadcast()` to Avoid Shuffle on Small Tables

When one table is very small (like `df_dept` with just 3 rows), you can use a **Broadcast Join**. Spark copies the small table to **every executor** so the join happens locally — **no shuffle**.

```python
from pyspark.sql.functions import broadcast

# df_dept is tiny → broadcast it!
df_final = df_step3.join(
    broadcast(df_dept),   # <-- copied to all executors
    on="department",
    how="inner"
)
```

### Why is this faster?

| Regular Join | Broadcast Join |
|-------------|----------------|
| Both tables shuffle across network | Only big table stays put |
| Expensive | Cheap |
| Slow | Fast ⚡ |

> **When to use:** If the small table is under **10 MB** (default), Spark may auto-broadcast it. But it is good practice to explicitly use `broadcast()` when you know a table is tiny.

---

## 5. Quick Reference: Shuffle Triggers

| Operation | Shuffles? | Reason |
|-----------|-----------|--------|
| `filter()` / `where()` | ❌ No | Row-level check |
| `withColumn()` / `select()` | ❌ No | Column-level change |
| `drop()` / `cast()` | ❌ No | Schema change only |
| `groupBy()` + `agg()` | ✅ **Yes** | Same key must meet |
| `join()` (regular) | ✅ **Yes** | Matching keys must meet |
| `join()` (broadcast) | ❌ No | Small table copied everywhere |
| `repartition()` | ✅ **Yes** | Explicitly redistributes data |
| `orderBy()` / `sort()` | ✅ **Yes** | Global sort needs all data |
| `distinct()` | ⚠️ Sometimes | May need shuffle to find uniqueness |

---

## 6. Key Takeaways 

1. **Shuffle = Data movement across the network.** It is the most expensive thing in Spark.
2. **Narrow Transformations** (`filter`, `withColumn`) are cheap and fast because each executor works alone.
3. **Wide Transformations** (`groupBy`, `join`, `sort`) are expensive because they need data from multiple executors to meet.
4. **`df_dept` is a lookup table.** It adds extra information (budget) to your main data using a `join`.
5. **Use `broadcast()`** when joining a big table with a tiny table. It eliminates shuffle and makes your job fly.
6. **Always check `df.explain()`** to see where shuffles ("Exchange" operators) appear in your physical plan.

---

## 7. How to Read `explain()` Output

When you run `df_final.explain(extended=True)`, look for these keywords:

| Keyword | Meaning |
|---------|---------|
| `Exchange` | **SHUFFLE is happening here!** |
| `HashAggregate` | `groupBy()` + `agg()` |
| `SortMergeJoin` | Regular join (may shuffle) |
| `BroadcastHashJoin` | Broadcast join (no shuffle) 🎉 |
| `Filter` | `filter()` / `where()` |
| `Project` | `select()` / `withColumn()` |

> **Tip:** If you see too many `Exchange` nodes, your job is doing too much shuffling. Try to reduce them with `broadcast()` or by filtering early.

---




### 3. Advanced Tasks 
## Task 8

In [0]:
# ============================================
# IMPORTS & SAMPLE DATA CREATION
# ============================================
from pyspark.sql.functions import (
    col,
    month,
    year,
    sum as spark_sum,
    count,
    avg,
    to_date,
    lit,
    when,
    round,
    date_format,
    concat_ws
)

# Create sample sales data — simulates a real e-commerce dataset
data = [
    ("2024-01-15", "Electronics", "Laptop", 1200.00, 2, "USA"),
    ("2024-01-20", "Electronics", "Phone", 800.00, 5, "USA"),
    ("2024-02-10", "Clothing", "Jacket", 150.00, 10, "UK"),
    ("2024-02-15", "Electronics", "Tablet", 400.00, 3, "USA"),
    ("2024-03-05", "Clothing", "Shirt", 50.00, 20, "UK"),
    ("2024-03-12", "Home", "Chair", 200.00, 4, "USA"),
    ("2024-03-20", "Electronics", "Laptop", 1200.00, 1, "Germany"),
    ("2024-04-08", "Home", "Table", 350.00, 2, "UK"),
    ("2024-04-15", "Clothing", "Shoes", 120.00, 8, "Germany"),
    ("2024-05-01", "Electronics", "Phone", 800.00, 3, "USA"),
    ("2024-05-20", "Home", "Lamp", 80.00, 15, "UK"),
    ("2024-06-10", "Clothing", "Jacket", 150.00, 5, "Germany"),
    ("2024-06-25", "Electronics", "Tablet", 400.00, 4, "USA"),
    ("2024-07-05", "Home", "Chair", 200.00, 6, "UK"),
    ("2024-07-18", "Electronics", "Laptop", 1200.00, 2, "Germany"),
    ("2024-08-01", "Clothing", "Shirt", 50.00, 25, "USA"),
    ("2024-08-15", "Home", "Table", 350.00, 3, "UK"),
    ("2024-09-10", "Electronics", "Phone", 800.00, 4, "Germany"),
    ("2024-09-25", "Clothing", "Shoes", 120.00, 10, "USA"),
    ("2024-10-05", "Home", "Lamp", 80.00, 20, "UK"),
    ("2024-10-20", "Electronics", "Laptop", 1200.00, 3, "Germany"),
    ("2024-11-01", "Clothing", "Jacket", 150.00, 7, "USA"),
    ("2024-11-15", "Home", "Chair", 200.00, 5, "UK"),
    ("2024-12-01", "Electronics", "Tablet", 400.00, 6, "Germany"),
    ("2024-12-20", "Clothing", "Shirt", 50.00, 30, "USA")
]

columns = ["date", "category", "product", "unit_price", "quantity", "country"]

# Create Spark DataFrame
df_spark = spark.createDataFrame(data , columns)

# Convert date string to actual DateType
df_spark = df_spark.withColumn("date" , to_date(col("date") , "yyyy-MM-dd"))

# Add a revenue column (unit_price * quantity)
df_spark = df_spark.withColumn("revenue" , round(col("unit_price") * col("quantity"), 2))

print("Sample data created successfully!")
df_spark.show(5)
print(f"Total Rows  : {df_spark.count()}")

In [0]:
# ============================================
# PANDAS APPROACH (For Comparison)
# ============================================
import pandas as pd

# Convert Spark DataFrame to pandas (WARNING: Only works for small data!)
df_pandas = df_spark.toPandas()

df_pandas['month'] = pd.to_datetime(df_pandas['date']).dt.month
df_pandas['year'] = pd.to_datetime(df_pandas['date']).dt.year

result_pandas = df_pandas.groupby(['year' , 'month' , 'category']).agg({
    'revenue' : 'sum',
    'quantity' : "sum"
}).reset_index()

print("Pandas Result (works only on single machine memory):")
print("=" * 80)
print(result_pandas)

In [0]:
# ============================================
# PYSPARK APPROACH (Scalable)
# ============================================

# PySpark approach: Same logic, but distributed across a cluster

result_spark = (
    df_spark
    .withColumn("year" , year(col("date")))
    .withColumn("month" , month(col("date")))
    .groupBy("year" , "month" , "category")
    .agg(
        spark_sum("revenue").alias("total_revenue"),
        spark_sum("quantity").alias("total_quantity")
    )
    .orderBy("year" , "month" , "category")
)

print("PySpark Result (scales to billions of rows):")
result_spark.show()

# ⚡ Task 7: Why Pandas Fails & Spark Wins

## 📚 The Library Analogy

Imagine you are tasked with **counting every single book in a massive university library**:

* 👤 **Pandas (Single Worker):** You walk through every single shelf alone, carrying all the books in your arms at the same time. If the stack of books gets too heavy for your arms, you drop everything (**Crash/Out of Memory**). If you trip and fall halfway through, you have to stand up and recount the entire library from scratch (**No Fault Tolerance**).
* 👷‍♂️ **PySpark (Team of Helpers):** You hire a team of 100 helpers (**Distributed Nodes**). Each helper counts just one shelf simultaneously (**Parallel Processing**). If one helper gets sick or trips, the rest of the team keeps working, and another worker takes over that single broken shelf (**Fault Tolerance**).

---

## 🛠️ The 3 Core Differences

| Feature | 🐼 Pandas | ⚡ PySpark |
| :--- | :--- | :--- |
| **1. Memory (RAM)** | **Single Machine Bound:** Loads the entire dataset into your local machine's RAM. If `Dataset Size > RAM Size`, it crashes instantly with an `OutOfMemoryError`. | **Distributed Across Cluster:** Splits large datasets into smaller fragments (*Partitions*) and distributes them across the RAM of many computers. |
| **2. Speed & Execution** | **Single-Core Execution:** Operations run sequentially on a single CPU core, leaving the rest of your system's processing power unused. | **Multi-Core Parallel Processing:** Divides tasks across hundreds of CPU cores simultaneously, reducing computation time from hours to seconds. |
| **3. Reliability & Recovery** | **All-or-Nothing:** If a process fails or a machine disconnects at 99%, the entire job dies and you must restart from 0%. | **Resilient & Self-Healing:** Keeps track of how data was transformed (*Lineage Graph / DAG*). If a single node crashes, Spark recalculates **only the lost partition** without restarting the job. |

## Task 9

In [0]:
# ============================================
# PARTITIONING STRATEGY DESIGN
# ============================================

# First, let us add year and month columns for partitioning
df_partitioned = (
    df_spark
    .withColumn("year" ,  year(col("date")))
    .withColumn("month" , month(col("date")))
)

# WRITE STRATEGY: Partition by year AND month
# This creates a folder structure like:
#   /sales_partitioned_by_date/
#     ├── year=2024/
#     │     ├── month=1/
#     │     │     └── part-00001.parquet
#     │     ├── month=2/
#     │     │     └── part-00001.parquet
#     │     └── ...
#     └── year=2025/
#           └── ...

df_partitioned.write\
    .mode("overwrite")\
    .partitionBy("year" , "month")\
    .parquet("/Volumes/cyntexa_dev/spark_basics_day4/raw")

print("Data written with partitioning strategy!")

# Show the physical folder structure
import subprocess 
result = subprocess.run(["ls" , "-R" , "/Volumes/cyntexa_dev/spark_basics_day4/raw/year=2024/"] , capture_output = True , text = True)
print("\nPhysical folder structure created:")
print(result.stdout[:1500])  # Show first 1500 chars


# 🎯 Task 8: Partitioning Strategy Justification

---

## 💼 The Business Problem
Our business sales data is primarily queried by **DATE RANGES** (e.g., *"Show me Q1 2024"*, *"Last 30 days"*, or *"December 2024 revenue"*). 

Without a smart storage strategy, every simple query scans the entire dataset from scratch, making queries **slow** and cloud computing bills **expensive**.

---

## 🗂️ Our Strategy: `partitionBy("year", "month")`

### 📁 How Data is Structured on Disk (Hive-Style Partitioning)
Instead of putting billions of sales rows into one giant, messy file, Spark automatically organizes data into subfolders like a digital filing cabinet:

```text
sales_data/
├── year=2024/
│   ├── month=01/   <-- January 2024 data only
│   ├── month=02/   <-- February 2024 data only
│   ├── month=03/   <-- March 2024 data only
│   └── ...
└── year=2025/
    ├── month=01/   <-- January 2025 data only
    └── ...
```

---

## 💡 4 Reasons Why This Strategy is Smart

### 1. Partition Pruning *(Skip What You Don't Need)*
Imagine looking for a specific receipt from January 2024 in a massive office:
* ❌ **Without Partitioning:** You must open and read **every single document** in the entire building just to find January 2024 records.
* ✅ **With Partitioning:** You walk straight to the folder labeled **`year=2024/month=01/`** and pull only those pages. You completely ignore the other 11 months (and other years).

> 🚀 **Business Impact:** Queries run **10x to 1000x faster**, and cloud storage costs drop significantly because we read far fewer gigabytes of data.

---

### 2. Lazy Evaluation & Optimizer Shortcuts *(Plan Before You Execute)*
Spark doesn't run code blindly. When you write a request:

```python
# Filtering for January 2024
df.filter((col("year") == 2024) & (col("month") == 1))
```

Spark’s internal engine (**Catalyst Optimizer**) checks the folder structure *before* executing:
1. *"Aha! The user wants Year 2024 and Month 1."*
2. *"This table is partitioned by year and month!"*
3. *"I can skip 95% of the folders on disk before reading a single file."*

> 🧠 **Analogy:** It’s like a smart librarian who knows the exact shelf location of a book instead of searching room-by-room.

---

### 3. Target File Size *(The Goldilocks Rule)*
For maximum speed, file sizes within each folder must be **"just right"**:

| ❌ Too Small (< 10 MB) | ❌ Too Big (> 2 GB) | ✅ Just Right (128 MB – 256 MB) |
| :--- | :--- | :--- |
| Creates massive overhead ("Small File Problem"). Like reading 1,000 sticky notes. | Hard to split across worker computers; causes slow single-thread bottlenecks. | Optimal balance for parallel processing and fast reads. |

> 💡 **Best Practice:** We enable Spark's **AQE (Adaptive Query Execution)** or use `repartition()` before saving data to hit the **~128MB sweet spot** per file.

---

### 4. Physical Execution Plan *(The Evidence)*
We can verify that Spark is skipping unnecessary folders by inspecting the **Physical Execution Plan**:

```python
# Inspecting the execution plan in Spark
df.filter((col("year") == 2024) & (col("month") == 1)).explain()
```

#### What to look for in the plan output:
```text
PartitionFilters: [isnotnull(year#0), isnotnull(month#1), (year#0 = 2024), (month#1 = 1)]
ReadSchema: struct<category:string, revenue:double, quantity:int>
```

When you see `PartitionFilters` in the plan, it confirms **Partition Pruning is active**—Spark is opening *only* the matching folder and completely ignoring the rest of the storage.


In [0]:
# ============================================
# PROOF — PARTITION PRUNING IN ACTION
# ============================================

# Read back the partitioned data
partitioned_df = spark.read.parquet("/Volumes/cyntexa_dev/spark_basics_day4/raw")

# Query: Only January 2024 data
january_query = partitioned_df.filter(
    (col("year") == 2024) & (col("month") == 1)
)

print("-" * 70)
january_query.explain()
print("-" * 70)

print("\nIf you see PartitionFilters: [isnotnull(year), (year = 2024), ...]")
print("   then Spark is ONLY reading the January 2024 folder!")


## Task 9

In [0]:
# ============================================
# TASK 9 — MONTHLY REVENUE BY CATEGORY REPORT
# ============================================

# Step 1: Extract year and month from the date
df_report = (
    df_spark
    .withColumn("year" , year(col("date")))
    .withColumn("month" , month(col("date")))
    .withColumn("month_name" , date_format(col("date") , "MMMM"))

)

# Step 2: Group by Year, Month, and Category — calculate totals
monthly_revenue_report = (
    df_report
    .groupBy("year" , "month" , "month_name" , "category")
    .agg(
        spark_sum("revenue").alias("total_revenue"),
        spark_sum("quantity").alias("units_sold"),
        count("product").alias("num_transactions"),
        round(avg("unit_price") , 2).alias("avg_unit_price")
    )
    .orderBy("year" , "month" , "category")
)

print("MONTHLY REVENUE BY CATEGORY REPORT")
print("=" * 70)
monthly_revenue_report.show(30, truncate=False)

In [0]:
# Step 3: Write report to Databricks Catalog / Delta Table

monthly_revenue_report.write\
    .mode("overwrite")\
    .saveAsTable("cyntexa_dev.spark_basics_day4.monthly_revenue_by_category")

print("Report successfully saved as Delta Table: 'monthly_revenue_by_category'")